# EDA + Análisis del Modelo — Throw-In Predictor

Notebook para explorar el dataset de modelado y analizar el modelo entrenado.

**Pre-requisitos**:
- `python -m model.dataset_builder` ejecutado → `data/model/dataset.parquet`
- `python -m model.train` ejecutado → `data/model/model_v1.joblib` + `feature_importance.csv` + `metrics_v1.json`

Secciones:
1. Carga y exploración básica del dataset
2. Distribución del target
3. Verificación anti-leakage (rolling mira solo fechas anteriores)
4. Verificación de reproducibilidad del dataset
5. Métricas del modelo entrenado
6. Feature importance + ablation top-N
7. Análisis de residuos

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

DATASET = Path('../data/model/dataset.parquet')
MODEL = Path('../data/model/model_v1.joblib')
METRICS = Path('../data/model/metrics_v1.json')
FI = Path('../data/model/feature_importance.csv')

## 1. Carga y exploración

In [ ]:
df = pd.read_parquet(DATASET)
print('shape:', df.shape)
print('seasons:')
print(df.groupby('season').size())
print('\nis_home balance:')
print(df['is_home'].value_counts())
df.head(3)

## 2. Distribución del target

In [ ]:
print(df['throw_ins_total'].describe())
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df['throw_ins_total'].hist(bins=40, ax=ax[0])
ax[0].set_title('Distribución throw_ins_total (por equipo)')
ax[0].set_xlabel('throw_ins_total')
by_home = df.groupby('is_home')['throw_ins_total'].mean()
by_home.plot(kind='bar', ax=ax[1])
ax[1].set_title('Media por is_home')
ax[1].set_xticklabels(['Away', 'Home'], rotation=0)
plt.tight_layout()
plt.show()

## 3. Verificación anti-leakage

Para 3 filas aleatorias, comprobamos que las features rolling solo miran partidos con fecha anterior.

In [ ]:
sample = df.sample(3, random_state=7).sort_values('match_date')
for _, row in sample.iterrows():
    print(f"\n--- match_id={row['match_id']} team={row['team_name']} date={row['match_date'].date()} is_home={row['is_home']}")
    history = df[(df['team_id'] == row['team_id']) & (df['match_date'] < row['match_date'])].sort_values('match_date').tail(5)
    print('  últimos 5 partidos previos del equipo:')
    print(history[['match_date', 'throw_ins_total']].to_string(index=False))
    print(f"  rolling5_throw_ins_total esperado: {history['throw_ins_total'].mean():.3f}")
    print(f"  rolling5_throw_ins_total en fila : {row['rolling5_throw_ins_total']:.3f}")

## 4. Reproducibilidad del dataset

Regenerar y comparar que el resultado es idéntico.

In [ ]:
# Descomentar para verificar (tarda ~2s):
# from model.dataset_builder import build_dataset
# df2 = build_dataset()
# df_sorted = df.sort_values(['match_id', 'is_home']).reset_index(drop=True)
# df2_sorted = df2.sort_values(['match_id', 'is_home']).reset_index(drop=True)
# num_cols = df.select_dtypes(include=[np.number]).columns
# pd.testing.assert_frame_equal(df_sorted[num_cols], df2_sorted[num_cols])
# print('Reproducibilidad OK')

## 5. Métricas del modelo

In [ ]:
with open(METRICS, encoding='utf-8') as f:
    metrics = json.load(f)
print(json.dumps({
    'best_model': metrics['best_model'],
    'baseline_mae': round(metrics['baseline_mae'], 4),
    'beats_baseline': metrics['beats_baseline'],
    'n_features': metrics['n_features'],
    'train_rows': metrics['train_rows'],
    'val_rows': metrics['val_rows'],
}, indent=2))
print('\nResumen por configuración:')
for name, r in metrics['results'].items():
    print(f"  {name}: MAE {r['mae']:.4f}  RMSE {r['rmse']:.4f}")

## 6. Feature importance + ablation

In [ ]:
fi = pd.read_csv(FI)
top = fi.head(25)
plt.figure(figsize=(8, 8))
plt.barh(top['feature'][::-1], top['importance_gain'][::-1])
plt.xlabel('importance_gain')
plt.title('Top 25 features por gain')
plt.tight_layout()
plt.show()

print('\nCategorías más importantes por tipo de feature:')
def categorize(name):
    if name.startswith('opp_'): return 'opponent'
    if name.startswith('rolling'): return 'rolling_self'
    if name.startswith('ewma_'): return 'ewma_self'
    if name.startswith('std_'): return 'season_to_date'
    return 'other'
fi['category'] = fi['feature'].apply(categorize)
print(fi.groupby('category')['importance_gain'].agg(['sum', 'mean', 'count']).sort_values('sum', ascending=False))

## 7. Análisis de residuos (validación 2024/25)

In [ ]:
artifact = joblib.load(MODEL)
model = artifact['model']
features = artifact['features']

val = df[df['season'] == '2024/2025'].copy()
val['pred'] = model.predict(val[features])
val['residual'] = val['throw_ins_total'] - val['pred']

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
val['residual'].hist(bins=40, ax=ax[0])
ax[0].set_title(f"Residuos val — mean {val['residual'].mean():.2f} std {val['residual'].std():.2f}")
ax[0].axvline(0, color='red', linestyle='--')

by_team = val.groupby('team_name').apply(lambda d: (d['throw_ins_total'] - d['pred']).abs().mean()).sort_values(ascending=False)
by_team.head(10).plot(kind='barh', ax=ax[1])
ax[1].set_title('MAE por equipo (peores 10 en val)')
plt.tight_layout()
plt.show()